# Hafta 7 — Hiperparametre Optimizasyonu ve Kalibrasyon

Arama `src/tuning.py`, kalibrasyon ve aday seçimi kuralları `src/final_model.py` içindedir. Rapordaki sayıların kaynağı `notebooks/build_hiperparametre.py` ile üretilen `hafta7_ozet.json` dosyasıdır (30 yapılandırma, iki aday). Bu notebook yöntemi XGBoost ile 5 yapılandırmalık küçük bir aramayla gösterir. Test seti ayrılır ama hiçbir hücrede kullanılmaz.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_raw
from src.evaluate import calibration_summary
from src.modeling import build_model, cross_validate_model, make_cv
from src.preprocessing import split_train_test
from src.tuning import PARAM_SPECS, describe_spec, most_influential_params, tune_model

split = split_train_test(load_raw())
print('eğitim:', split.X_train.shape, 'test:', split.X_test.shape)

eğitim: (36168, 15) test: (9043, 15)


## 1. Arama uzayı

In [2]:
pd.Series({param: describe_spec(spec) for param, spec in PARAM_SPECS['xgboost'].items()})

model__n_estimators           tamsayı [200, 800 (hariç)]
model__learning_rate       logaritmik düzgün [0.01, 0.2]
model__max_depth                  tamsayı [3, 9 (hariç)]
model__min_child_weight          tamsayı [1, 10 (hariç)]
model__subsample                       düzgün [0.6, 1.0]
model__colsample_bytree                düzgün [0.5, 1.0]
model__reg_lambda            logaritmik düzgün [0.1, 10]
dtype: str

## 2. XGBoost için 5 yapılandırmalık küçük arama (PR-AUC'ye göre)

In [3]:
search = tune_model(build_model('xgboost'), PARAM_SPECS['xgboost'], split.X_train, split.y_train, n_iter=5)
print('en iyi parametreler:', search.best_params)
print('en iyi CV PR-AUC:', round(search.best_score, 4))
print('en etkili parametreler:', most_influential_params(search.cv_results))

en iyi parametreler: {'model__colsample_bytree': 0.7159725093210578, 'model__learning_rate': 0.023927528765580644, 'model__max_depth': 5, 'model__min_child_weight': 3, 'model__n_estimators': 766, 'model__reg_lambda': 9.25681899206688, 'model__subsample': 0.786705157299192}
en iyi CV PR-AUC: 0.4563
en etkili parametreler: ['model__min_child_weight', 'model__learning_rate']


## 3. Kalibrasyon özeti (ayarlanmış model, OOF tahminleri)

In [4]:
tuned = cross_validate_model(build_model('xgboost').set_params(**search.best_params), split.X_train, split.y_train, make_cv())
summary = calibration_summary(split.y_train, tuned.oof_proba)
print('Brier:', round(summary['brier'], 5))
pd.DataFrame(summary['bins']).round(4)

Brier: 0.08139


,mean_predicted,observed_rate,count
0,0.0206,0.0229,3617
1,0.0312,0.0346,3617
2,0.0396,0.0359,3617
3,0.0485,0.0448,3617
4,0.0582,0.0556,3617
5,0.0699,0.0683,3617
6,0.0857,0.0749,3617
7,0.1114,0.1070,3617
8,0.1878,0.2190,3616
9,0.5115,0.5069,3616
